## Installing required libraries

In [1]:
!pip install -q -U numpy chromadb llama-index llama-index-embeddings-huggingface llama-index-vector-stores-chroma llama-index-readers-file pypdf transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 62.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 95.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 93.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 94.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 78.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 10.5 MB/s eta 0:00:00
   

In [2]:
!pip install -q llama-index-readers-file

## Chunking and Storing

In [3]:
#initiating a client and collection 
import chromadb

db         = chromadb.PersistentClient(path='./chroma_db')
collection = db.get_or_create_collection('whitepapers')

In [4]:
#connect it with llamaindex
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.core import StorageContext

vector_store = ChromaVectorStore(chroma_collection=collection)
storage_context      = StorageContext.from_defaults(vector_store=vector_store)

Now llamaindex writes into chromadb directly 

In [5]:
from llama_index.core import VectorStoreIndex, Settings
from llama_index.readers.file import PDFReader
from pathlib import Path

dataset = Path('/kaggle/input/datasets/adwaittagalpallewar/kaggle-whitepaper-pdfs')

loader = PDFReader()

documents = []

for pdf in dataset.glob('*.pdf'):
    doc = loader.load_data(file=pdf)
    documents.extend(doc)

print(documents[0].text[:500])

2026-05-29 12:16:52,707 - INFO - NumExpr defaulting to 4 threads.


Agents
Authors: Julia Wiesinger, Patrick Marlow  
and Vladimir Vuskovic



In [6]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.embed_model = HuggingFaceEmbedding(
    model_name="BAAI/bge-small-en-v1.5"
)

index = VectorStoreIndex(
    documents,
    storage_context=storage_context
)

2026-05-29 12:17:24,099 - INFO - TensorFlow version 2.19.0 available.
2026-05-29 12:17:24,101 - INFO - JAX version 0.7.2 available.
2026-05-29 12:17:32,973 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:17:32,975 - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-05-29 12:17:32,991 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-05-29 12:17:33,011 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

2026-05-29 12:17:33,088 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:17:33,104 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-29 12:17:33,121 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"


config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

2026-05-29 12:17:33,134 - INFO - Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-05-29 12:17:33,191 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:17:33,207 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-05-29 12:17:33,267 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:17:33,283 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/README.md "HTTP/1.1 200 OK"
2026-05-29 12:17:33,299 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a

...

2026-05-29 12:17:37,658 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-05-29 12:17:37,720 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:17:37,736 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/special_tokens_map.json "HTTP/1.1 200 OK"
2026-05-29 12:17:37,753 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

2026-05-29 12:17:37,828 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"
2026-05-29 12:17:37,989 - INFO - HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/1_Pooling/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:17:38,005 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"
2026-05-29 12:17:38,022 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/1_Pooling%2Fconfig.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

2026-05-29 12:17:38,099 - INFO - HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"
2026-05-29 12:17:38,506 - INFO - Loaded 1 prompt with these keys: ['query']


In [7]:
print(collection.count()) #number of nodes/chunks stored in the db

260


In [8]:
for doc in documents:
    print(doc.metadata)
    break

{'page_label': '1', 'file_name': '22365_19_Agents_v8.pdf'}


In [9]:
collection.get(include=["documents"],where={'page_label':'23'}) #this is simply us querying with the db

{'ids': ['1296c8d6-519c-4935-b086-c34ff29f51e7',
  '815a8100-e0cf-4b2c-8c31-48606b9e93ac',
  '337313b5-640b-4d3d-adc6-659e58787c55',
  '38812c3e-4009-4c11-ab8f-56b2a46f1a7b'],
 'embeddings': None,
 'documents': ["Agents\n23\nFebruary 2025\n \nFigure 9. Sequence diagram showing the lifecycle of a Function Call\nThe result of the example in Figure 9 is that the model is leveraged to “fill in the blanks” with \nthe parameters required for the Client side UI to make the call to the Google Places API. The \nClient side UI manages the actual API call using the parameters provided by the model in the \nreturned Function. This is just one use case for Function Calling, but there are many other \nscenarios to consider like:\n• You want a language model to suggest a function that you can use in your code, but you \ndon't want to include credentials in your code. Because function calling doesn't run the \nfunction, you don't need to include credentials in your code with the function information."

## Multi-query retrival

In [10]:
from transformers import pipeline

generator = pipeline(
    'text-generation',
    model='TinyLlama/TinyLlama-1.1B-Chat-v1.0',
    device_map='auto'
)

retriever = index.as_retriever(
    similarity_top_k = 2
)

2026-05-29 12:30:36,862 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:30:36,879 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
2026-05-29 12:30:36,898 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

2026-05-29 12:30:37,020 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"
2026-05-29 12:30:37,100 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:30:37,117 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/config.json "HTTP/1.1 200 OK"
2026-05-29 12:30:37,187 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/model.safetensors "HTTP/1.1 302 Found"
2026-05-29 12:30:37,251 - INFO - HTTP Request: GET https://huggingface.co/api/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/xet-read-token/fe8a4ea1ffedaf415f4da2f062534de366a451e6 "HTTP/1.1 200 OK"


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

2026-05-29 12:30:47,599 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/tokenizer.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:30:47,616 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer.json "HTTP/1.1 200 OK"
2026-05-29 12:30:47,637 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/tokenizer.json "HTTP/1.1 200 OK"


tokenizer.json: 0.00B [00:00, ?B/s]

2026-05-29 12:30:47,765 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/added_tokens.json "HTTP/1.1 404 Not Found"
2026-05-29 12:30:47,832 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/special_tokens_map.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:30:47,848 - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/special_tokens_map.json "HTTP/1.1 200 OK"
2026-05-29 12:30:47,866 - INFO - HTTP Request: GET https://huggingface.co/api/resolve-cache/models/TinyLlama/TinyLlama-1.1B-Chat-v1.0/fe8a4ea1ffedaf415f4da2f062534de366a451e6/special_tokens_map.json "HTTP/1.1 200 OK"


special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

2026-05-29 12:30:47,942 - INFO - HTTP Request: HEAD https://huggingface.co/TinyLlama/TinyLlama-1.1B-Chat-v1.0/resolve/main/chat_template.jinja "HTTP/1.1 404 Not Found"


In [26]:
prompt = """
Generate 3 different search queries related to:
'What is prompt engineering?'

Return only the queries.
"""

generated_queries = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

#ertrieval
all_nodes = []

for q in generated_queries[0]['generated_text']:
    nodes = retriever.retrieve(q)
    all_nodes.extend(nodes)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [27]:
print(generated_queries[0]['generated_text'])


Generate 3 different search queries related to:
'What is prompt engineering?'

Return only the queries.

Examples:

1. What is prompt engineering?
2. What is the best way to promote prompt engineering?
3. How can I promote prompt engineering in my company?
4. What are the benefits of prompt engineering?
5. What are the challenges of prompt engineering?
6. How can I improve prompt engineering in my team?
7. What are the key factors for promoting prompt engineering?
8. How can I measure the success of prompt


In [30]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)

query = "What is prompt engineering?"

pairs = [
    [query,node.text]
    for node in all_nodes
]

scores = reranker.predict(pairs)

ranked = sorted(
    zip(all_nodes, scores),
    key=lambda x: x[1],
    reverse=True
)

for node, score in ranked:
    print(score)
    print(node.text[:500])
    print("="*50)
    break

2026-05-29 12:50:09,446 - INFO - No device provided, using cuda:0
2026-05-29 12:50:09,591 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:50:09,651 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 404 Not Found"
2026-05-29 12:50:09,653 - INFO - No modules.json found for cross-encoder/ms-marco-MiniLM-L-6-v2, initializing a new CrossEncoder model.
2026-05-29 12:50:09,712 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L-6-v2 "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:50:09,774 - INFO - HTTP Request: GET https://huggingface.co/api/models/cross-encoder/ms-marco-MiniLM-L6-v2 "HTTP/1.1 200 OK"
2026-05-29 12:50:09,835 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

2026-05-29 12:50:10,486 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/processor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:50:10,550 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-05-29 12:50:10,614 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:50:10,676 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-05-29 12:50:10,737 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-MiniLM-L-6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 307 Temporary Redirect"
2026-05-29 12:50:10,803 - INFO - HTTP Request: HEAD https://huggingface.co/cross-encoder/ms-marco-Mi

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

2.9672232
Prompt Engineering
February 2025
23
Here are some styles you can choose from which I find effective:
Confrontational, Descriptive, Direct, Formal, Humorous, Influential, Informal, 
Inspirational, Persuasive
Let’s change our prompt in Table 6 to include a humorous and inspirational style.
Prompt I want you to act as a travel guide. I will write to you about 
my location and you will suggest 3 places to visit near me in 
a humorous style.
My suggestion: "I am in Manhattan."
Travel Suggestions:
Out


Well the retrieved chunk is about prompt engineering but it really doesnt answer or contain any relevent content about our query thats what i think.